In [12]:
import requests
import pandas as pd

In [13]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import os

# CONFIGURACIÓN MANUAL (Sustituye esto por tu clave real)
os.environ['API_KEY_C'] = 'TU_CLAVE_AQUI_ENTRE_COMILLAS'

# Ahora el resto de tu código ya puede leerla
api_key = os.getenv('API_KEY_C')

if api_key:
    print(f" Clave lista para usar: {api_key[:5]}...")

 Clave lista para usar: TU_CL...


## DANA

In [ ]:
import requests
import time

# Configuración
api_key =  #VUETRA API KEY
estacion = "8414A"  # Aeropuerto de Valencia
fecha_inicio = "2024-10-28T00:00:00UTC"
fecha_fin = "2024-10-31T23:59:59UTC"

url = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_inicio}/fechafin/{fecha_fin}/estacion/{estacion}"

headers = {'cache-control': "no-cache"}
params = {"api_key": api_key}

# PASO 1: Solicitar el enlace de los datos
print("Solicitando enlace de descarga a AEMET...")
res1 = requests.get(url, headers=headers, params=params)
meta_datos = res1.json()

if meta_datos.get("estado") == 200:
    url_descarga = meta_datos.get("datos")
    print(f"Enlace obtenido. Descargando datos finales...")
    
    # Esperar un segundo por seguridad (rate limiting)
    time.sleep(1)
    
    # PASO 2: Descargar los datos reales (aquí es donde fallaba antes)
    res2 = requests.get(url_descarga)
    
    if res2.status_code == 200:
        datos_dana = res2.json()
        
        # Mostrar resultados (ejemplo: lluvia acumulada)
        for dia in datos_dana:
            fecha = dia.get('fecha')
            precipitacion = dia.get('prec', '0')
            print(f"Fecha: {fecha} | Precipitación: {precipitacion} mm")
    else:
        print("Error: El enlace de descarga caducó demasiado rápido.")
else:
    print(f"Error en la petición: {meta_datos.get('descripcion')}")

Solicitando enlace de descarga a AEMET...
Enlace obtenido. Descargando datos finales...
Fecha: 2024-10-28 | Precipitación: 8,0 mm
Fecha: 2024-10-29 | Precipitación: 6,6 mm
Fecha: 2024-10-30 | Precipitación: 0,0 mm
Fecha: 2024-10-31 | Precipitación: 0,0 mm


In [17]:
datos_dana

[{'fecha': '2024-10-28',
  'indicativo': '8414A',
  'nombre': 'VALENCIA AEROPUERTO',
  'provincia': 'VALENCIA',
  'altitud': '56',
  'tmed': '17,0',
  'prec': '8,0',
  'tmin': '12,0',
  'horatmin': '05:54',
  'tmax': '22,0',
  'horatmax': '13:08',
  'dir': '06',
  'velmedia': '3,6',
  'racha': '9,7',
  'horaracha': '15:46',
  'sol': '5,2',
  'presMax': '1014,2',
  'horaPresMax': 'Varias',
  'presMin': '1010,9',
  'horaPresMin': '01',
  'hrMedia': '83',
  'hrMax': '97',
  'horaHrMax': 'Varias',
  'hrMin': '66',
  'horaHrMin': '12:35'},
 {'fecha': '2024-10-29',
  'indicativo': '8414A',
  'nombre': 'VALENCIA AEROPUERTO',
  'provincia': 'VALENCIA',
  'altitud': '56',
  'tmed': '19,6',
  'prec': '6,6',
  'tmin': '18,1',
  'horatmin': '06:50',
  'tmax': '21,2',
  'horatmax': 'Varias',
  'dir': '99',
  'velmedia': '9,7',
  'racha': '22,2',
  'horaracha': 'Varias',
  'sol': '0,0',
  'presMax': '1013,6',
  'horaPresMax': '01',
  'hrMedia': '92'},
 {'fecha': '2024-10-30',
  'indicativo': '8414A'

In [18]:
datos_aeropuerto = [...] # Aquí va tu lista de diccionarios

print(f"{'FECHA':<12} | {'TEMP MEDIA':<10} | {'PRECIPITACIÓN':<10} | {'RACHA MAX'}")
print("-" * 55)

for dia in datos_dana:
    # Cambiamos la coma por el punto para que Python lo entienda como número
    tmed = float(dia['tmed'].replace(',', '.'))
    prec = float(dia['prec'].replace(',', '.'))
    # En el día 29 algunos datos pueden faltar por el temporal, usamos .get()
    racha = dia.get('racha', 'N/D').replace(',', '.')
    
    print(f"{dia['fecha']:<12} | {tmed:<10} | {prec:<10} | {racha} km/h")

FECHA        | TEMP MEDIA | PRECIPITACIÓN | RACHA MAX
-------------------------------------------------------
2024-10-28   | 17.0       | 8.0        | 9.7 km/h
2024-10-29   | 19.6       | 6.6        | 22.2 km/h
2024-10-30   | 20.6       | 0.0        | 17.5 km/h
2024-10-31   | 19.9       | 0.0        | 8.9 km/h


In [19]:
df_dana = pd.DataFrame(datos_dana)
df_dana

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,...,sol,presMax,horaPresMax,presMin,horaPresMin,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin
0,2024-10-28,8414A,VALENCIA AEROPUERTO,VALENCIA,56,"17,0","8,0","12,0",05:54,"22,0",...,"5,2","1014,2",Varias,"1010,9",01,83,97,Varias,66,12:35
1,2024-10-29,8414A,VALENCIA AEROPUERTO,VALENCIA,56,"19,6","6,6","18,1",06:50,"21,2",...,"0,0","1013,6",01,NaN,NaN,92,NaN,NaN,NaN,NaN
2,2024-10-30,8414A,VALENCIA AEROPUERTO,VALENCIA,56,"20,6","0,0","17,4",23:59,"23,7",...,"7,2","1011,9",Varias,"1005,6",02,87,96,Varias,69,11:42
3,2024-10-31,8414A,VALENCIA AEROPUERTO,VALENCIA,56,"19,9","0,0","16,1",03:30,"23,7",...,"6,0","1015,3",Varias,"1011,4",03,75,95,Varias,60,13:06


In [ ]:
import requests
import time

# Configuración
api_key =  #VUESTRA API KEY
estacion = "8096"  # utiel
fecha_inicio = "2024-10-28T00:00:00UTC"
fecha_fin = "2024-10-31T23:59:59UTC"

url = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_inicio}/fechafin/{fecha_fin}/estacion/{estacion}"

headers = {'cache-control': "no-cache"}
params = {"api_key": api_key}

# PASO 1: Solicitar el enlace de los datos
print("Solicitando enlace de descarga a AEMET...")
res1 = requests.get(url, headers=headers, params=params)
meta_datos = res1.json()

if meta_datos.get("estado") == 200:
    url_descarga = meta_datos.get("datos")
    print(f"Enlace obtenido. Descargando datos finales...")
    
    # Esperar un segundo por seguridad (rate limiting)
    time.sleep(1)
    
    # PASO 2: Descargar los datos reales (aquí es donde fallaba antes)
    res2 = requests.get(url_descarga)
    
    if res2.status_code == 200:
        datos_dana_utiel = res2.json()
        
        # Mostrar resultados (ejemplo: lluvia acumulada)
        for dia in datos_dana_utiel:
            fecha = dia.get('fecha')
            precipitacion1 = dia.get('prec', '0')
            print(f"Fecha: {fecha} | Precipitación: {precipitacion} mm")
    else:
        print("Error: El enlace de descarga caducó demasiado rápido.")
else:
    print(f"Error en la petición: {meta_datos.get('descripcion')}")

Solicitando enlace de descarga a AEMET...
Enlace obtenido. Descargando datos finales...
Fecha: 2024-10-28 | Precipitación: 0,0 mm
Fecha: 2024-10-29 | Precipitación: 0,0 mm
Fecha: 2024-10-30 | Precipitación: 0,0 mm
Fecha: 2024-10-31 | Precipitación: 0,0 mm


In [42]:
datos_dana_utiel

[{'fecha': '2024-10-28',
  'indicativo': '8096',
  'nombre': 'CUENCA',
  'provincia': 'CUENCA',
  'altitud': '949',
  'tmed': '13,6',
  'prec': '0,4',
  'tmin': '7,7',
  'horatmin': '05:50',
  'tmax': '19,5',
  'horatmax': '15:10',
  'dir': '06',
  'velmedia': '2,2',
  'racha': '9,2',
  'horaracha': '23:10',
  'sol': '7,7',
  'presMax': '912,3',
  'horaPresMax': '11',
  'presMin': '909,4',
  'horaPresMin': '03',
  'hrMedia': '73',
  'hrMax': '96',
  'horaHrMax': '01:10',
  'hrMin': '51',
  'horaHrMin': '14:20'},
 {'fecha': '2024-10-29',
  'indicativo': '8096',
  'nombre': 'CUENCA',
  'provincia': 'CUENCA',
  'altitud': '949',
  'tmed': '12,6',
  'prec': '52,2',
  'tmin': '9,9',
  'horatmin': '23:59',
  'tmax': '15,3',
  'horatmax': 'Varias',
  'dir': '06',
  'velmedia': '4,2',
  'racha': '15,3',
  'horaracha': '02:40',
  'sol': '0,3',
  'presMax': '912,3',
  'horaPresMax': '00',
  'presMin': '905,7',
  'horaPresMin': 'Varias',
  'hrMedia': '83',
  'hrMax': '93',
  'horaHrMax': 'Varias'

In [43]:
df_dana_utiel = pd.DataFrame(datos_dana_utiel)
df_dana_utiel

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,...,sol,presMax,horaPresMax,presMin,horaPresMin,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin
0,2024-10-28,8096,CUENCA,CUENCA,949,"13,6","0,4","7,7",05:50,"19,5",...,"7,7","912,3",11,"909,4",03,73,96,01:10,51,14:20
1,2024-10-29,8096,CUENCA,CUENCA,949,"12,6","52,2","9,9",23:59,"15,3",...,"0,3","912,3",00,"905,7",Varias,83,93,Varias,63,Varias
2,2024-10-30,8096,CUENCA,CUENCA,949,"14,6","0,0","9,8",00:20,"19,5",...,"8,4","910,9",24,"905,9",06,60,100,04:50,36,13:20
3,2024-10-31,8096,CUENCA,CUENCA,949,"14,9","0,0","10,0",07:10,"19,8",...,"7,0","913,6",22,"910,8",01,65,83,Varias,45,13:30


## FILOMENA

In [ ]:
import requests
import time

# Configuración
api_key =  #VUESTRA API KEY
estacion = "3195"  # Madrid Retiro
fecha_inicio = "2021-01-07T00:00:00UTC"
fecha_fin = "2021-01-12T23:59:59UTC"

url = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_inicio}/fechafin/{fecha_fin}/estacion/{estacion}"

headers = {'cache-control': "no-cache"}
params = {"api_key": api_key}

# PASO 1: Solicitar el enlace de los datos
print("Solicitando enlace de descarga a AEMET...")
res1 = requests.get(url, headers=headers, params=params)
meta_datos = res1.json()

if meta_datos.get("estado") == 200:
    url_descarga = meta_datos.get("datos")
    print(f"Enlace obtenido. Descargando datos finales...")
    
    # Esperar un segundo por seguridad (rate limiting)
    time.sleep(1)
    
    # PASO 2: Descargar los datos reales (aquí es donde fallaba antes)
    res2 = requests.get(url_descarga)
    
    if res2.status_code == 200:
        datos_filomena = res2.json()
        
        # Mostrar resultados (ejemplo: lluvia acumulada)
        for dia in datos_filomena:
            fecha = dia.get('fecha')
            precipitacion1 = dia.get('prec', '0')
            print(f"Fecha: {fecha} | Precipitación: {precipitacion} mm")
    else:
        print("Error: El enlace de descarga caducó demasiado rápido.")
else:
    print(f"Error en la petición: {meta_datos.get('descripcion')}")

Solicitando enlace de descarga a AEMET...
Enlace obtenido. Descargando datos finales...
Fecha: 2021-01-07 | Precipitación: 0,0 mm
Fecha: 2021-01-08 | Precipitación: 0,0 mm
Fecha: 2021-01-09 | Precipitación: 0,0 mm
Fecha: 2021-01-10 | Precipitación: 0,0 mm
Fecha: 2021-01-11 | Precipitación: 0,0 mm
Fecha: 2021-01-12 | Precipitación: 0,0 mm


In [28]:
datos_filomena

[{'fecha': '2021-01-07',
  'indicativo': '3195',
  'nombre': 'MADRID, RETIRO',
  'provincia': 'MADRID',
  'altitud': '667',
  'tmed': '0,4',
  'prec': '2,3',
  'tmin': '-0,6',
  'horatmin': '22:30',
  'tmax': '1,4',
  'horatmax': 'Varias',
  'presMax': '935,1',
  'horaPresMax': '10',
  'presMin': '932,8',
  'horaPresMin': '05',
  'hrMedia': '82',
  'hrMax': '97',
  'horaHrMax': '14:30',
  'hrMin': '57',
  'horaHrMin': '00:00'},
 {'fecha': '2021-01-08',
  'indicativo': '3195',
  'nombre': 'MADRID, RETIRO',
  'provincia': 'MADRID',
  'altitud': '667',
  'tmed': '-0,4',
  'prec': '32,9',
  'tmin': '-1,0',
  'horatmin': 'Varias',
  'tmax': '0,3',
  'horatmax': 'Varias',
  'presMax': '933,7',
  'horaPresMax': 'Varias',
  'presMin': '926,5',
  'horaPresMin': '24',
  'hrMedia': '87',
  'hrMax': '99',
  'horaHrMax': 'Varias',
  'hrMin': '67',
  'horaHrMin': 'Varias'},
 {'fecha': '2021-01-09',
  'indicativo': '3195',
  'nombre': 'MADRID, RETIRO',
  'provincia': 'MADRID',
  'altitud': '667',
  '

In [29]:
# Si datos_filomena es una lista que contiene OTRA lista con los datos:
if isinstance(datos_filomena[0], list):
    datos_procesar = datos_filomena[0]
else:
    datos_procesar = datos_filomena

print(f"{'FECHA':<12} | {'T. MIN':<8} | {'T. MAX':<8} | {'PRECIP (mm)':<10}")
print("-" * 50)

for dia in datos_procesar:
    # Verificamos que 'dia' sea un diccionario y no otra cosa
    if isinstance(dia, dict):
        fecha = dia.get('fecha', 'N/A')
        tmin = float(dia.get('tmin', '0').replace(',', '.'))
        tmax = float(dia.get('tmax', '0').replace(',', '.'))
        prec = float(dia.get('prec', '0').replace(',', '.'))
        
        print(f"{fecha:<12} | {tmin:>6}°C | {tmax:>6}°C | {prec:>10.1f}")

FECHA        | T. MIN   | T. MAX   | PRECIP (mm)
--------------------------------------------------
2021-01-07   |   -0.6°C |    1.4°C |        2.3
2021-01-08   |   -1.0°C |    0.3°C |       32.9
2021-01-09   |   -1.6°C |    0.5°C |       17.7
2021-01-10   |   -0.8°C |    3.2°C |        0.0
2021-01-11   |   -4.4°C |    3.2°C |        0.0
2021-01-12   |   -7.4°C |    0.7°C |        0.0


In [ ]:
import requests
import time

# Configuración
api_key =  #VUESTRA API KEY
estacion = "3260B"  # Toledo
fecha_inicio = "2021-01-07T00:00:00UTC"
fecha_fin = "2021-01-12T23:59:59UTC"

url = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_inicio}/fechafin/{fecha_fin}/estacion/{estacion}"

headers = {'cache-control': "no-cache"}
params = {"api_key": api_key}

# PASO 1: Solicitar el enlace de los datos
print("Solicitando enlace de descarga a AEMET...")
res1 = requests.get(url, headers=headers, params=params)
meta_datos = res1.json()

if meta_datos.get("estado") == 200:
    url_descarga = meta_datos.get("datos")
    print(f"Enlace obtenido. Descargando datos finales...")
    
    # Esperar un segundo por seguridad (rate limiting)
    time.sleep(1)
    
    # PASO 2: Descargar los datos reales (aquí es donde fallaba antes)
    res2 = requests.get(url_descarga)
    
    if res2.status_code == 200:
        datos_filomena_toledo = res2.json()
        
        # Mostrar resultados (ejemplo: lluvia acumulada)
        for dia in datos_filomena_toledo:
            fecha = dia.get('fecha')
            precipitacion1 = dia.get('prec', '0')
            print(f"Fecha: {fecha} | Precipitación: {precipitacion} mm")
    else:
        print("Error: El enlace de descarga caducó demasiado rápido.")
else:
    print(f"Error en la petición: {meta_datos.get('descripcion')}")

Solicitando enlace de descarga a AEMET...
Enlace obtenido. Descargando datos finales...
Fecha: 2021-01-07 | Precipitación: 0,0 mm
Fecha: 2021-01-08 | Precipitación: 0,0 mm
Fecha: 2021-01-09 | Precipitación: 0,0 mm
Fecha: 2021-01-10 | Precipitación: 0,0 mm
Fecha: 2021-01-11 | Precipitación: 0,0 mm
Fecha: 2021-01-12 | Precipitación: 0,0 mm


In [32]:
datos_filomena_toledo

[{'fecha': '2021-01-07',
  'indicativo': '3260B',
  'nombre': 'TOLEDO',
  'provincia': 'TOLEDO',
  'altitud': '513',
  'tmed': '0,8',
  'prec': '9,2',
  'tmin': '-0,7',
  'horatmin': '23:59',
  'tmax': '2,3',
  'horatmax': 'Varias',
  'dir': '07',
  'velmedia': '2,8',
  'racha': '7,2',
  'horaracha': '02:00',
  'sol': '0,0',
  'presMax': '951,9',
  'horaPresMax': 'Varias',
  'presMin': '949,5',
  'horaPresMin': '04',
  'hrMedia': '98',
  'hrMax': '100',
  'horaHrMax': 'Varias',
  'hrMin': '71',
  'horaHrMin': 'Varias'},
 {'fecha': '2021-01-08',
  'indicativo': '3260B',
  'nombre': 'TOLEDO',
  'provincia': 'TOLEDO',
  'altitud': '513',
  'tmed': '0,0',
  'prec': '36,6',
  'tmin': '-1,1',
  'horatmin': '05:00',
  'tmax': '1,0',
  'horatmax': '13:00',
  'dir': '07',
  'velmedia': '4,4',
  'racha': '10,6',
  'horaracha': '16:50',
  'sol': '0,0',
  'presMax': '951,0',
  'horaPresMax': '00',
  'presMin': '942,0',
  'horaPresMin': '24',
  'hrMedia': '97',
  'hrMax': '100',
  'horaHrMax': 'Var

In [33]:
# Si datos_filomena es una lista que contiene OTRA lista con los datos:
if isinstance(datos_filomena_toledo[0], list):
    datos_procesar = datos_filomena_toledo[0]
else:
    datos_procesar = datos_filomena_toledo

print(f"{'FECHA':<12} | {'T. MIN':<8} | {'T. MAX':<8} | {'PRECIP (mm)':<10}")
print("-" * 50)

for dia in datos_procesar:
    # Verificamos que 'dia' sea un diccionario y no otra cosa
    if isinstance(dia, dict):
        fecha = dia.get('fecha', 'N/A')
        tmin = float(dia.get('tmin', '0').replace(',', '.'))
        tmax = float(dia.get('tmax', '0').replace(',', '.'))
        prec = float(dia.get('prec', '0').replace(',', '.'))
        
        print(f"{fecha:<12} | {tmin:>6}°C | {tmax:>6}°C | {prec:>10.1f}")

FECHA        | T. MIN   | T. MAX   | PRECIP (mm)
--------------------------------------------------
2021-01-07   |   -0.7°C |    2.3°C |        9.2
2021-01-08   |   -1.1°C |    1.0°C |       36.6
2021-01-09   |   -2.4°C |    1.8°C |        9.0
2021-01-10   |   -4.1°C |    4.3°C |        0.0
2021-01-11   |   -8.0°C |    5.2°C |        0.0
2021-01-12   |  -13.4°C |    0.9°C |        0.0


In [34]:
df_filomena = pd.DataFrame(datos_filomena)
df_filomena

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,horatmax,presMax,horaPresMax,presMin,horaPresMin,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin
0,2021-01-07,3195,"MADRID, RETIRO",MADRID,667,"0,4","2,3","-0,6",22:30,"1,4",Varias,"935,1",10,"932,8",05,82,97,14:30,57,00:00
1,2021-01-08,3195,"MADRID, RETIRO",MADRID,667,"-0,4","32,9","-1,0",Varias,"0,3",Varias,"933,7",Varias,"926,5",24,87,99,Varias,67,Varias
2,2021-01-09,3195,"MADRID, RETIRO",MADRID,667,"-0,6","17,7","-1,6",Varias,"0,5",21:10,"930,1",23,"924,1",13,99,100,Varias,98,Varias
3,2021-01-10,3195,"MADRID, RETIRO",MADRID,667,"1,2","0,0","-0,8",05:10,"3,2",14:30,"938,9",23,"930,0",00,86,99,Varias,75,14:30
4,2021-01-11,3195,"MADRID, RETIRO",MADRID,667,"-0,6","0,0","-4,4",07:40,"3,2",13:40,"947,5",24,"938,8",00,70,95,23:20,57,14:00
5,2021-01-12,3195,"MADRID, RETIRO",MADRID,667,"-3,4","0,0","-7,4",08:20,"0,7",13:40,"951,8",23,"947,5",00,76,93,00:00,61,12:10


In [35]:
df_filomena_toledo = pd.DataFrame(datos_filomena_toledo)
df_filomena_toledo

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,...,sol,presMax,horaPresMax,presMin,horaPresMin,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin
0,2021-01-07,3260B,TOLEDO,TOLEDO,513,"0,8","9,2","-0,7",23:59,"2,3",...,"0,0","951,9",Varias,"949,5",04,98,100,Varias,71,Varias
1,2021-01-08,3260B,TOLEDO,TOLEDO,513,"0,0","36,6","-1,1",05:00,"1,0",...,"0,0","951,0",00,"942,0",24,97,100,Varias,86,01:40
2,2021-01-09,3260B,TOLEDO,TOLEDO,513,"-0,3","9,0","-2,4",23:30,"1,8",...,"0,0",NaN,NaN,NaN,NaN,98,100,Varias,92,15:00
3,2021-01-10,3260B,TOLEDO,TOLEDO,513,"0,1",NaN,"-4,1",23:20,"4,3",...,"7,0",NaN,NaN,NaN,NaN,94,100,Varias,74,15:20
4,2021-01-11,3260B,TOLEDO,TOLEDO,513,"-1,4",NaN,"-8,0",08:00,"5,2",...,"8,9","965,5",24,"956,7",00,80,100,03:30,57,13:40
5,2021-01-12,3260B,TOLEDO,TOLEDO,513,"-6,2",NaN,"-13,4",08:00,"0,9",...,"8,8","969,8",23,"965,3",00,82,98,09:00,65,15:00


In [ ]:
# 1. Lista de todos tus DataFrames
lista_estaciones = [df_dana, df_dana_utiel]

# 2. Unirlos todos de golpe
df_total_evento = pd.concat(lista_estaciones, ignore_index=True)

# 3. Ordenar por fecha para que el trabajo se vea profesional
df_total_evento_dana = df_total_evento.sort_values(by='fecha')

In [47]:
df_total_evento_dana

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,...,sol,presMax,horaPresMax,presMin,horaPresMin,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin
0,2024-10-28,8414A,VALENCIA AEROPUERTO,VALENCIA,56,"17,0","8,0","12,0",05:54,"22,0",...,"5,2","1014,2",Varias,"1010,9",01,83,97,Varias,66,12:35
4,2024-10-28,8096,CUENCA,CUENCA,949,"13,6","0,4","7,7",05:50,"19,5",...,"7,7","912,3",11,"909,4",03,73,96,01:10,51,14:20
1,2024-10-29,8414A,VALENCIA AEROPUERTO,VALENCIA,56,"19,6","6,6","18,1",06:50,"21,2",...,"0,0","1013,6",01,NaN,NaN,92,NaN,NaN,NaN,NaN
5,2024-10-29,8096,CUENCA,CUENCA,949,"12,6","52,2","9,9",23:59,"15,3",...,"0,3","912,3",00,"905,7",Varias,83,93,Varias,63,Varias
2,2024-10-30,8414A,VALENCIA AEROPUERTO,VALENCIA,56,"20,6","0,0","17,4",23:59,"23,7",...,"7,2","1011,9",Varias,"1005,6",02,87,96,Varias,69,11:42
6,2024-10-30,8096,CUENCA,CUENCA,949,"14,6","0,0","9,8",00:20,"19,5",...,"8,4","910,9",24,"905,9",06,60,100,04:50,36,13:20
3,2024-10-31,8414A,VALENCIA AEROPUERTO,VALENCIA,56,"19,9","0,0","16,1",03:30,"23,7",...,"6,0","1015,3",Varias,"1011,4",03,75,95,Varias,60,13:06
7,2024-10-31,8096,CUENCA,CUENCA,949,"14,9","0,0","10,0",07:10,"19,8",...,"7,0","913,6",22,"910,8",01,65,83,Varias,45,13:30


In [48]:
# 1. Lista de todos tus DataFrames
lista_estaciones = [df_filomena,df_filomena_toledo]

# 2. Unirlos todos de golpe
df_total = pd.concat(lista_estaciones, ignore_index=True)

# 3. Ordenar por fecha para que el trabajo se vea profesional
df_total_evento_filomena = df_total.sort_values(by='fecha')

In [49]:
df_total_evento_filomena

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,...,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin,dir,velmedia,racha,horaracha,sol
0,2021-01-07,3195,"MADRID, RETIRO",MADRID,667,"0,4","2,3","-0,6",22:30,"1,4",...,82,97,14:30,57,00:00,NaN,NaN,NaN,NaN,NaN
6,2021-01-07,3260B,TOLEDO,TOLEDO,513,"0,8","9,2","-0,7",23:59,"2,3",...,98,100,Varias,71,Varias,07,"2,8","7,2",02:00,"0,0"
1,2021-01-08,3195,"MADRID, RETIRO",MADRID,667,"-0,4","32,9","-1,0",Varias,"0,3",...,87,99,Varias,67,Varias,NaN,NaN,NaN,NaN,NaN
7,2021-01-08,3260B,TOLEDO,TOLEDO,513,"0,0","36,6","-1,1",05:00,"1,0",...,97,100,Varias,86,01:40,07,"4,4","10,6",16:50,"0,0"
2,2021-01-09,3195,"MADRID, RETIRO",MADRID,667,"-0,6","17,7","-1,6",Varias,"0,5",...,99,100,Varias,98,Varias,NaN,NaN,NaN,NaN,NaN
8,2021-01-09,3260B,TOLEDO,TOLEDO,513,"-0,3","9,0","-2,4",23:30,"1,8",...,98,100,Varias,92,15:00,05,"2,5","10,3",05:50,"0,0"
3,2021-01-10,3195,"MADRID, RETIRO",MADRID,667,"1,2","0,0","-0,8",05:10,"3,2",...,86,99,Varias,75,14:30,NaN,NaN,NaN,NaN,NaN
9,2021-01-10,3260B,TOLEDO,TOLEDO,513,"0,1",NaN,"-4,1",23:20,"4,3",...,94,100,Varias,74,15:20,27,"0,8","2,8",12:10,"7,0"
4,2021-01-11,3195,"MADRID, RETIRO",MADRID,667,"-0,6","0,0","-4,4",07:40,"3,2",...,70,95,23:20,57,14:00,NaN,NaN,NaN,NaN,NaN
10,2021-01-11,3260B,TOLEDO,TOLEDO,513,"-1,4",NaN,"-8,0",08:00,"5,2",...,80,100,03:30,57,13:40,99,"0,3","2,5",14:50,"8,9"


In [50]:
# 1. Guardar el DataFrame de la DANA
# 'index=False' evita que se guarde una columna extra con los números de fila
# 'encoding='utf-8-sig'' es el truco para que Excel reconozca bien los acentos españoles
df_total_evento_dana.to_csv('datos_dana_unificados.csv', index=False, encoding='utf-8-sig', sep=';')

# 2. Guardar el DataFrame de Filomena
df_total_evento_filomena.to_csv('datos_filomena_unificados.csv', index=False, encoding='utf-8-sig', sep=';')

print("✅ Archivos CSV creados correctamente.")

✅ Archivos CSV creados correctamente.


In [ ]:

# Nos aseguramnos que las fechas estan igual que en dataframe
datos_emergencias = {
    '2021-01-08': 12500, '2021-01-09': 21500, '2021-01-10': 18000,
    '2024-10-28': 500, '2024-10-29': 30000, '2024-10-30': 12000
}

def integrar_datos_emergencia(df):
    # Hacemos una copia para no alterar el original por error
    nuevo_df = df.copy()
    
    # PASO CLAVE: Limpiar la columna fecha de posibles espacios o formatos raros
    nuevo_df['fecha'] = nuevo_df['fecha'].astype(str).str.strip()
    
    # Creamos la columna usando .map()
    nuevo_df['llamadas_112'] = nuevo_df['fecha'].map(datos_emergencias)
    
    # Si después del map todo son NaN, es que las fechas no coinciden
    # Rellenamos con 0 para que la columna exista sí o sí
    nuevo_df['llamadas_112'] = nuevo_df['llamadas_112'].fillna(0).astype(int)
    
    return nuevo_df

# IMPORTANTE: Tienes que asignar el resultado a la variable de nuevo
df_total_dana = integrar_datos_emergencia(df_total_evento_dana)
df_total_filomena = integrar_datos_emergencia(df_total_evento_filomena)

# Comprobación inmediata
print("Columnas actuales en DANA:", df_total_dana.columns)

Columnas actuales en DANA: Index(['fecha', 'indicativo', 'nombre', 'provincia', 'altitud', 'tmed', 'prec',
       'tmin', 'horatmin', 'tmax', 'horatmax', 'dir', 'velmedia', 'racha',
       'horaracha', 'sol', 'presMax', 'horaPresMax', 'presMin', 'horaPresMin',
       'hrMedia', 'hrMax', 'horaHrMax', 'hrMin', 'horaHrMin', 'llamadas_112'],
      dtype='str')


In [59]:
df_total_dana

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,...,presMax,horaPresMax,presMin,horaPresMin,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin,llamadas_112
0,2024-10-28,8414A,VALENCIA AEROPUERTO,VALENCIA,56,"17,0","8,0","12,0",05:54,"22,0",...,"1014,2",Varias,"1010,9",01,83,97,Varias,66,12:35,500
4,2024-10-28,8096,CUENCA,CUENCA,949,"13,6","0,4","7,7",05:50,"19,5",...,"912,3",11,"909,4",03,73,96,01:10,51,14:20,500
1,2024-10-29,8414A,VALENCIA AEROPUERTO,VALENCIA,56,"19,6","6,6","18,1",06:50,"21,2",...,"1013,6",01,NaN,NaN,92,NaN,NaN,NaN,NaN,30000
5,2024-10-29,8096,CUENCA,CUENCA,949,"12,6","52,2","9,9",23:59,"15,3",...,"912,3",00,"905,7",Varias,83,93,Varias,63,Varias,30000
2,2024-10-30,8414A,VALENCIA AEROPUERTO,VALENCIA,56,"20,6","0,0","17,4",23:59,"23,7",...,"1011,9",Varias,"1005,6",02,87,96,Varias,69,11:42,12000
6,2024-10-30,8096,CUENCA,CUENCA,949,"14,6","0,0","9,8",00:20,"19,5",...,"910,9",24,"905,9",06,60,100,04:50,36,13:20,12000
3,2024-10-31,8414A,VALENCIA AEROPUERTO,VALENCIA,56,"19,9","0,0","16,1",03:30,"23,7",...,"1015,3",Varias,"1011,4",03,75,95,Varias,60,13:06,0
7,2024-10-31,8096,CUENCA,CUENCA,949,"14,9","0,0","10,0",07:10,"19,8",...,"913,6",22,"910,8",01,65,83,Varias,45,13:30,0


In [60]:
df_total_filomena

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,...,hrMax,horaHrMax,hrMin,horaHrMin,dir,velmedia,racha,horaracha,sol,llamadas_112
0,2021-01-07,3195,"MADRID, RETIRO",MADRID,667,"0,4","2,3","-0,6",22:30,"1,4",...,97,14:30,57,00:00,NaN,NaN,NaN,NaN,NaN,0
6,2021-01-07,3260B,TOLEDO,TOLEDO,513,"0,8","9,2","-0,7",23:59,"2,3",...,100,Varias,71,Varias,07,"2,8","7,2",02:00,"0,0",0
1,2021-01-08,3195,"MADRID, RETIRO",MADRID,667,"-0,4","32,9","-1,0",Varias,"0,3",...,99,Varias,67,Varias,NaN,NaN,NaN,NaN,NaN,12500
7,2021-01-08,3260B,TOLEDO,TOLEDO,513,"0,0","36,6","-1,1",05:00,"1,0",...,100,Varias,86,01:40,07,"4,4","10,6",16:50,"0,0",12500
2,2021-01-09,3195,"MADRID, RETIRO",MADRID,667,"-0,6","17,7","-1,6",Varias,"0,5",...,100,Varias,98,Varias,NaN,NaN,NaN,NaN,NaN,21500
8,2021-01-09,3260B,TOLEDO,TOLEDO,513,"-0,3","9,0","-2,4",23:30,"1,8",...,100,Varias,92,15:00,05,"2,5","10,3",05:50,"0,0",21500
3,2021-01-10,3195,"MADRID, RETIRO",MADRID,667,"1,2","0,0","-0,8",05:10,"3,2",...,99,Varias,75,14:30,NaN,NaN,NaN,NaN,NaN,18000
9,2021-01-10,3260B,TOLEDO,TOLEDO,513,"0,1",NaN,"-4,1",23:20,"4,3",...,100,Varias,74,15:20,27,"0,8","2,8",12:10,"7,0",18000
4,2021-01-11,3195,"MADRID, RETIRO",MADRID,667,"-0,6","0,0","-4,4",07:40,"3,2",...,95,23:20,57,14:00,NaN,NaN,NaN,NaN,NaN,0
10,2021-01-11,3260B,TOLEDO,TOLEDO,513,"-1,4",NaN,"-8,0",08:00,"5,2",...,100,03:30,57,13:40,99,"0,3","2,5",14:50,"8,9",0


In [61]:
# 1. Guardar el DataFrame de la DANA
# 'index=False' evita que se guarde una columna extra con los números de fila
# 'encoding='utf-8-sig'' es el truco para que Excel reconozca bien los acentos españoles
df_total_dana.to_csv('datos_dana_emergencias.csv', index=False, encoding='utf-8-sig', sep=';')

# 2. Guardar el DataFrame de Filomena
df_total_filomena.to_csv('datos_filomena_emergencias.csv', index=False, encoding='utf-8-sig', sep=';')

print("✅ Archivos CSV creados correctamente.")

✅ Archivos CSV creados correctamente.


In [ ]:
import pandas as pd

In [3]:
df_avisos = pd.read_csv("avisos_aemet_2021_2026_moderate_severe_extreme.csv")

In [5]:
df_avisos.head(5)

,onset,expires,año,mes,mes_num,estacion,event,severity,riesgo_numerico,duracion_horas,area_desc
0,2021-01-01 06:00:00+00:00,2021-01-01 19:59:59+00:00,2021,Enero,1,Invierno,Aviso de vientos de nivel amarillo,Moderate,1,13.999722,Valle del Almanzora y Los Vélez
1,2021-01-01 06:00:00+00:00,2021-01-01 19:59:59+00:00,2021,Enero,1,Invierno,Moderate wind warning,Moderate,1,13.999722,Valle del Almanzora y Los Vélez
2,2021-01-01 06:00:00+00:00,2021-01-01 19:59:59+00:00,2021,Enero,1,Invierno,Aviso de vientos de nivel amarillo,Moderate,1,13.999722,Nacimiento y Campo de Tabernas
3,2021-01-01 06:00:00+00:00,2021-01-01 19:59:59+00:00,2021,Enero,1,Invierno,Moderate wind warning,Moderate,1,13.999722,Nacimiento y Campo de Tabernas
4,2021-01-01 06:00:00+00:00,2021-01-01 20:59:59+00:00,2021,Enero,1,Invierno,Aviso de vientos de nivel amarillo,Moderate,1,14.999722,Poniente y Almería Capital


onset', 'expires', 'año', 'mes', 'mes_num', 'estacion', 'event', 'severity', 'riesgo_numerico', 'duracion_horas', 'area_desc']Conteo severity:{'Moderate': 281140, 'Severe': 43018, 'Extreme': 1648}

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as ptl